# AllSci Application Crawler & Field Mapper

This notebook performs comprehensive crawling of the entire AllSci application to:
1. **Discover all pages** (list pages, detail pages, tabs)
2. **Build application sitemap** (structure and relationships)
3. **Map all data fields** on every page and tab
4. **Generate comprehensive documentation** of the application structure

In [4]:
pip install BeautifulSoup

  Preparing metadata (setup.py) ... error
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [7 lines of output]
      Traceback (most recent call last):
        File "<string>", line 2, in <module>
        File "<pip-setuptools-caller>", line 35, in <module>
        File "/private/var/folders/ft/1328jrdx05s3bnd035tpxmb80000gn/T/pip-install-_m6if2gs/beautifulsoup_bb4b4e886dfc42cabe5ebdcf06a71571/setup.py", line 3
          "You're trying to run a very old release of Beautiful Soup under Python 3. This will not work."<>"Please use Beautiful Soup 4, available through the pip package 'beautifulsoup4'."
                                                                                                         ^^
      SyntaxError: invalid syntax
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urlparse, urljoin, parse_qs
import json
from datetime import datetime
from collections import defaultdict, deque
import hashlib

ModuleNotFoundError: No module named 'bs4'

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Login credentials
SUPABASE_EMAIL = "rlalani@allsci.com"
SUPABASE_PASSWORD = "!!Casio1994$$"

# Base configuration
BASE_URL = "https://app.allsci.com"
LOGIN_URL = "https://app.allsci.com/?login=true"

# Search queries - starting points for crawling
# The application uses search-based navigation, not static list pages
SEARCH_QUERIES = [
    "covid",        # Example: finds hypotheses, articles, trials, grants, researchers
    "cancer",       # Different search term to discover different data
    # Add more search terms to discover different parts of the application
]

# Search result tabs to explore (these appear on search results)
SEARCH_RESULT_TABS = [
    'Hypotheses',
    'Articles',
    'Clinical Trials',
    'Grants',
    'Researchers',
]

# Additional navigation URLs (if any exist)
ADDITIONAL_URLS = [
    "https://app.allsci.com/explore/clinical-trials",  # Atlas view
    # Add other known URLs here
]

# Crawling limits
MAX_PAGES_TO_CRAWL = 50  # Limit total pages (increase for full crawl)
MAX_RESULTS_PER_SEARCH_TAB = 5  # How many detail pages to visit from each search result tab
MAX_DEPTH = 3  # How many levels deep to crawl

# Wait times
PAGE_LOAD_WAIT = 15  # seconds
ELEMENT_WAIT = 10    # seconds
TAB_SWITCH_WAIT = 3  # seconds
SEARCH_WAIT = 5      # seconds to wait for search results

# URL patterns to identify page types
PAGE_TYPE_PATTERNS = {
    'search_results': r'/search\?query=',
    'clinical_trial_detail': r'/clinical-trial/ASC-CT-\d+',
    'explore_atlas': r'/explore/clinical-trials',
    'work_detail': r'/work/ASC-WK-\d+',
    'patent_detail': r'/patent/ASC-PT-\d+',
    'hypothesis_detail': r'/hypothesis/ASC-HY-\d+',
    'grant_detail': r'/grant/ASC-GR-\d+',
    'researcher_detail': r'/researcher/ASC-RS-\d+',
}

# Detail page tab patterns (tabs that appear on detail pages like trials, works, etc.)
DETAIL_PAGE_TAB_PATTERNS = [
    'Overview',
    'Works',
    'Hypotheses',
    'Patents',
    'Related Trials',
    'Timeline',
    'Organizations',
    'People',
    'Funding',
]

## Helper Functions

In [ ]:
def setup_driver(headless=False):
    """Setup Chrome WebDriver."""
    chrome_options = Options()
    if headless:
        chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver


def login_to_application(driver, login_url, email, password):
    """Login to the application."""
    print(f"Logging in to: {login_url}")
    driver.get(login_url)
    
    try:
        email_input = WebDriverWait(driver, ELEMENT_WAIT).until(
            EC.presence_of_element_located((By.NAME, "email"))
        )
        email_input.send_keys(email)
        
        password_input = driver.find_element(By.NAME, "password")
        password_input.send_keys(password)
        
        sign_in_button = driver.find_element(By.XPATH, "//button[@type='submit' and contains(text(), 'Sign In')]")
        sign_in_button.click()
        
        time.sleep(5)
        print("✓ Login successful!")
        return True
        
    except Exception as e:
        print(f"✗ Login failed: {e}")
        return False


def wait_for_page_load(driver, timeout=PAGE_LOAD_WAIT):
    """Wait for page to finish loading."""
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script('return document.readyState') == 'complete'
        )
        time.sleep(2)
    except TimeoutException:
        pass


def normalize_url(url, base_url=BASE_URL):
    """Normalize URL for comparison."""
    # Remove fragments
    url = url.split('#')[0]
    # Remove trailing slashes for consistency
    url = url.rstrip('/')
    # Make absolute
    if not url.startswith('http'):
        url = urljoin(base_url, url)
    return url


def get_page_type(url):
    """Identify page type based on URL pattern."""
    for page_type, pattern in PAGE_TYPE_PATTERNS.items():
        if re.search(pattern, url):
            return page_type
    return 'unknown'


def get_url_hash(url):
    """Generate a hash for URL deduplication."""
    return hashlib.md5(normalize_url(url).encode()).hexdigest()[:12]


def get_css_selector(element, driver):
    """Generate a CSS selector for an element."""
    try:
        selector = driver.execute_script("""
            function getCssPath(el) {
                if (!(el instanceof Element)) return;
                var path = [];
                while (el.nodeType === Node.ELEMENT_NODE) {
                    var selector = el.nodeName.toLowerCase();
                    if (el.id) {
                        selector += '#' + el.id;
                        path.unshift(selector);
                        break;
                    } else {
                        var sib = el, nth = 1;
                        while (sib = sib.previousElementSibling) {
                            if (sib.nodeName.toLowerCase() == selector)
                                nth++;
                        }
                        if (nth != 1)
                            selector += ":nth-of-type("+nth+")";
                    }
                    path.unshift(selector);
                    el = el.parentNode;
                }
                return path.join(" > ");
            }
            return getCssPath(arguments[0]);
        """, element)
        return selector
    except:
        return "Unknown"

## Link Discovery Functions

In [ ]:
def discover_links(driver, base_url=BASE_URL):
    """Discover all internal links on the current page."""
    links = set()
    
    try:
        # Get all anchor tags
        anchor_elements = driver.find_elements(By.TAG_NAME, "a")
        
        for anchor in anchor_elements:
            try:
                href = anchor.get_attribute('href')
                if href:
                    # Normalize and check if internal
                    normalized = normalize_url(href, base_url)
                    if normalized.startswith(base_url):
                        links.add(normalized)
            except StaleElementReferenceException:
                continue
    except Exception as e:
        print(f"    Warning: Error discovering links: {e}")
    
    return links


def discover_tabs(driver):
    """Discover tabs/sections on detail pages (Overview, Works, Hypotheses, etc.)."""
    tabs = []
    
    # Common tab selectors
    tab_selectors = [
        "button[role='tab']",
        "a[role='tab']",
        "div[role='tab']",
        ".MuiTab-root",
        "[class*='tab']",
    ]
    
    for selector in tab_selectors:
        try:
            tab_elements = driver.find_elements(By.CSS_SELECTOR, selector)
            
            for tab in tab_elements:
                try:
                    tab_text = tab.text.strip()
                    if tab_text and len(tab_text) < 50:  # Reasonable tab name length
                        # Check if it's a known detail page tab pattern
                        if any(pattern.lower() in tab_text.lower() for pattern in DETAIL_PAGE_TAB_PATTERNS):
                            tabs.append({
                                'name': tab_text,
                                'element': tab,
                                'selector': selector
                            })
                except:
                    continue
        except:
            continue
    
    # Deduplicate by name
    seen_names = set()
    unique_tabs = []
    for tab in tabs:
        if tab['name'] not in seen_names:
            seen_names.add(tab['name'])
            unique_tabs.append(tab)
    
    return unique_tabs

In [ ]:
def perform_search(driver, query, base_url=BASE_URL):
    """Perform a search query and return the search results URL."""
    search_url = f"{base_url}/search?query={query}"
    print(f"  Performing search: '{query}'")
    driver.get(search_url)
    wait_for_page_load(driver)
    time.sleep(SEARCH_WAIT)
    return search_url


def discover_search_result_tabs(driver):
    """Discover search result tabs (Hypotheses, Articles, Clinical Trials, etc.)."""
    tabs = []
    
    try:
        # Look for tab elements in search results
        # Pattern: div with "flex flex-row cursor-pointer" containing tab name
        tab_elements = driver.find_elements(By.CSS_SELECTOR, "div.flex.flex-row.cursor-pointer")
        
        for tab_elem in tab_elements:
            try:
                # Get the tab name
                tab_name_elem = tab_elem.find_element(By.CSS_SELECTOR, "p.MuiTypography-root")
                tab_name = tab_name_elem.text.strip()
                
                # Get the count if available
                count_elem = tab_elem.find_element(By.CSS_SELECTOR, "p.MuiTypography-root + p")
                count = count_elem.text.strip() if count_elem else "0"
                
                if tab_name and tab_name in SEARCH_RESULT_TABS:
                    tabs.append({
                        'name': tab_name,
                        'count': count,
                        'element': tab_elem
                    })
            except:
                continue
    except Exception as e:
        print(f"    Warning: Could not discover search tabs: {e}")
    
    return tabs


def extract_filter_fields(driver):
    """Extract filter fields from the search results sidebar."""
    fields = []
    
    try:
        # Find accordion/filter sections
        accordions = driver.find_elements(By.CSS_SELECTOR, ".MuiAccordion-root")
        
        for accordion in accordions:
            try:
                # Get filter name
                filter_name_elem = accordion.find_element(By.CSS_SELECTOR, ".MuiAccordionSummary-content h1")
                filter_name = filter_name_elem.text.strip()
                
                # Get filter options
                options = accordion.find_elements(By.CSS_SELECTOR, "label")
                
                for option in options[:5]:  # Sample first 5 options
                    try:
                        option_text = option.text.strip()
                        if option_text:
                            # Split label and count if present
                            match = re.match(r'^(.+?)\s*\(([0-9,]+)\)$', option_text)
                            if match:
                                label = match.group(1)
                                count = match.group(2)
                            else:
                                label = option_text
                                count = ""
                            
                            fields.append({
                                'category': 'search_filter',
                                'label': f"{filter_name} - {label}",
                                'value': count,
                                'selector': get_css_selector(option, driver),
                                'element_type': 'filter',
                                'has_data': bool(count)
                            })
                    except:
                        continue
            except:
                continue
    except Exception as e:
        print(f"    Warning: Could not extract filters: {e}")
    
    return fields


def extract_search_result_metrics(driver):
    """Extract result counts from search tabs."""
    fields = []
    
    try:
        # Find tab elements with counts
        tab_divs = driver.find_elements(By.CSS_SELECTOR, "div.flex.flex-row.cursor-pointer")
        
        for tab_div in tab_divs:
            try:
                # Get tab name
                name_elem = tab_div.find_element(By.CSS_SELECTOR, "p.MuiTypography-root:first-child")
                tab_name = name_elem.text.strip()
                
                # Get count
                count_elem = tab_div.find_element(By.CSS_SELECTOR, "p.MuiTypography-root + p")
                count = count_elem.text.strip()
                
                if tab_name and count:
                    fields.append({
                        'category': 'search_result_count',
                        'label': tab_name,
                        'value': count,
                        'selector': get_css_selector(tab_div, driver),
                        'element_type': 'tab_metric',
                        'has_data': True
                    })
            except:
                continue
    except Exception as e:
        pass
    
    return fields


def crawl_search_results(driver, query, depth=0):
    """Crawl search results for a given query, including all tabs."""
    all_pages_data = []
    discovered_links = []
    
    # Perform search
    search_url = perform_search(driver, query)
    
    # Create page data for search results page
    search_page_data = {
        'url': search_url,
        'page_type': 'search_results',
        'title': driver.title,
        'depth': depth,
        'tabs': [],
        'fields': [],
        'links': set(),
        'crawl_timestamp': datetime.now().isoformat(),
        'search_query': query
    }
    
    # Extract search result metrics (tab counts)
    print(f"    Extracting search result metrics...")
    search_page_data['fields'].extend(extract_search_result_metrics(driver))
    
    # Extract filter fields
    print(f"    Extracting filter fields...")
    search_page_data['fields'].extend(extract_filter_fields(driver))
    
    # Add context to fields
    for field in search_page_data['fields']:
        field['page_url'] = search_url
        field['page_title'] = driver.title
        field['tab_name'] = None
        field['page_type'] = 'search_results'
    
    all_pages_data.append(search_page_data)
    
    # Discover and click through search result tabs
    tabs = discover_search_result_tabs(driver)
    if tabs:
        print(f"    Found {len(tabs)} search result tabs: {', '.join([t['name'] for t in tabs])}")
        
        for tab in tabs:
            try:
                print(f"      → Clicking search tab: {tab['name']} ({tab['count']} results)")
                tab['element'].click()
                time.sleep(TAB_SWITCH_WAIT)
                
                # Extract fields from this tab view
                tab_fields = extract_all_fields(driver, search_url, tab_name=tab['name'])
                search_page_data['fields'].extend(tab_fields)
                search_page_data['tabs'].append(tab['name'])
                
                # Discover links in this tab's results
                result_links = discover_links(driver)
                
                # Filter to only detail page links (not other search links)
                detail_links = [link for link in result_links 
                              if any(pattern in link for pattern in ['ASC-CT-', 'ASC-WK-', 'ASC-PT-', 'ASC-HY-', 'ASC-GR-', 'ASC-RS-'])]
                
                # Sample limited number of results
                sampled_links = detail_links[:MAX_RESULTS_PER_SEARCH_TAB]
                discovered_links.extend(sampled_links)
                
                print(f"        Found {len(detail_links)} detail pages, sampling {len(sampled_links)}")
                
            except Exception as e:
                print(f"        Warning: Could not process search tab {tab['name']}: {e}")
                continue
    
    print(f"    ✓ Search results page: {len(search_page_data['fields'])} fields, {len(discovered_links)} detail pages discovered")
    
    return all_pages_data, discovered_links

## Search-Based Navigation Functions

## Field Extraction Functions

In [ ]:
def extract_abstract_card(driver):    """Extract abstract card: trial abstract/description."""    fields = []        try:        # Look for Abstract section        abstract_elements = driver.find_elements(By.XPATH, "//h1[text()='Abstract']//following-sibling::p")                if abstract_elements:            abstract_text = ' '.join([elem.text.strip() for elem in abstract_elements])                        # Truncate if too long            if len(abstract_text) > 1000:                abstract_text = abstract_text[:1000] + "... [truncated]"                        fields.append({                'card': 'abstract',                'category': 'abstract_text',                'label': 'Trial Abstract',                'value': abstract_text,                'selector': get_css_selector(abstract_elements[0], driver),                'element_type': 'p',                'has_data': bool(abstract_text)            })    except Exception as e:        print(f"    Warning: Error extracting abstract card: {e}")        return fieldsdef extract_header_card_metrics(driver):    """Extract header card: trial type, status, title, organizations, etc."""    fields = []        try:        # Trial Type Badge        try:            type_badge = driver.find_element(By.XPATH, "//strong[contains(@class, 'uppercase') and contains(@class, 'tracking')]")            fields.append({                'card': 'header',                'category': 'trial_badge',                'label': 'Trial Type',                'value': type_badge.text.strip(),                'selector': get_css_selector(type_badge, driver),                'element_type': 'strong',                'has_data': True            })        except: pass                # Status Badge        try:            status_button = driver.find_element(By.XPATH, "//button/span[contains(text(), 'status')]")            fields.append({                'card': 'header',                'category': 'trial_status',                'label': 'Trial Status',                'value': status_button.text.strip(),                'selector': get_css_selector(status_button, driver),                'element_type': 'button',                'has_data': True            })        except: pass                # Trial Title        try:            title = driver.find_element(By.CSS_SELECTOR, "h2.text-\\[22px\\]")            fields.append({                'card': 'header',                'category': 'trial_title',                'label': 'Trial Title',                'value': title.text.strip(),                'selector': get_css_selector(title, driver),                'element_type': 'h2',                'has_data': True            })        except: pass                # Organizations        try:            org_elements = driver.find_elements(By.XPATH, "//svg[@viewBox='0 0 16 16']//following-sibling::div/span")            for org in org_elements[:1]:  # Get first org                fields.append({                    'card': 'header',                    'category': 'organization',                    'label': 'Primary Organization',                    'value': org.text.strip(),                    'selector': get_css_selector(org, driver),                    'element_type': 'span',                    'has_data': True                })        except: pass                # Location        try:            location = driver.find_element(By.XPATH, "//svg[@data-testid='LocationOnIcon']//following-sibling::span/span")            fields.append({                'card': 'header',                'category': 'location',                'label': 'Primary Location',                'value': location.text.strip(),                'selector': get_css_selector(location, driver),                'element_type': 'span',                'has_data': True            })        except: pass                # Source        try:            source_link = driver.find_element(By.XPATH, "//a[contains(@href, 'clinicaltrials.gov') or contains(@href, 'source')]")            fields.append({                'card': 'header',                'category': 'source',                'label': 'Data Source',                'value': source_link.get_attribute('href'),                'selector': get_css_selector(source_link, driver),                'element_type': 'a',                'has_data': True            })        except: pass            except Exception as e:        print(f"    Warning: Error extracting header card: {e}")        return fieldsdef extract_timeline_card(driver):    """Extract timeline card: key dates and milestones."""    fields = []        try:        # Look for timeline section        timeline_elements = driver.find_elements(By.XPATH, "//h1[text()='Timeline']//following-sibling::div//div[contains(@class, 'flex-row')]")                for elem in timeline_elements:            try:                # Get milestone name                milestone = elem.find_element(By.XPATH, ".//div[@class='flex flex-row gap-[7.6px] items-center ']").text.strip()                # Get date                date = elem.find_element(By.TAG_NAME, "span").text.strip()                                fields.append({                    'card': 'timeline',                    'category': 'timeline_event',                    'label': f'Timeline - {milestone}',                    'value': date,                    'selector': get_css_selector(elem, driver),                    'element_type': 'div',                    'has_data': True                })            except:                continue    except Exception as e:        print(f"    Warning: Error extracting timeline card: {e}")        return fieldsdef extract_metadata_card(driver):    """Extract metadata card: conditions, interventions, phase, enrollment."""    fields = []        try:        # Look for metadata section        metadata_section = driver.find_element(By.ID, "metadata-content")        metadata_items = metadata_section.find_elements(By.TAG_NAME, "h1")                for item in metadata_items:            text = item.text.strip()            if ':' in text:                label, value = text.split(':', 1)                label = label.strip()                value = value.strip()                                fields.append({                    'card': 'metadata',                    'category': 'metadata_field',                    'label': f'Metadata - {label}',                    'value': value,                    'selector': get_css_selector(item, driver),                    'element_type': 'h1',                    'has_data': bool(value)                })    except Exception as e:        print(f"    Warning: Error extracting metadata card: {e}")        return fieldsdef extract_study_design_card(driver):    """Extract study design card: purpose, allocation, model, masking."""    fields = []        try:        # Look for Study Design section        design_elements = driver.find_elements(By.XPATH, "//h1[text()='Study Design']//following-sibling::div//h1")                for elem in design_elements:            text = elem.text.strip()            if ':' in text:                label, value = text.split(':', 1)                label = label.strip()                value = value.strip()                                fields.append({                    'card': 'study_design',                    'category': 'design_parameter',                    'label': f'Study Design - {label}',                    'value': value,                    'selector': get_css_selector(elem, driver),                    'element_type': 'h1',                    'has_data': bool(value)                })    except Exception as e:        print(f"    Warning: Error extracting study design card: {e}")        return fieldsdef extract_participation_criteria_card(driver):    """Extract participation criteria: eligibility, age, sex, healthy volunteers."""    fields = []        try:        # Look for Participation Criteria section        criteria_elements = driver.find_elements(By.XPATH, "//h1[text()='Participation Criteria']//following-sibling::div//h1")                for elem in criteria_elements:            text = elem.text.strip()            if ':' in text:                label, value = text.split(':', 1)                label = label.strip()                value = value.strip()                                # Skip very long values (like full inclusion/exclusion lists)                if len(value) > 500:                    value = value[:500] + "... [truncated]"                                fields.append({                    'card': 'participation_criteria',                    'category': 'eligibility_parameter',                    'label': f'Eligibility - {label}',                    'value': value,                    'selector': get_css_selector(elem, driver),                    'element_type': 'h1',                    'has_data': bool(value)                })    except Exception as e:        print(f"    Warning: Error extracting participation criteria card: {e}")        return fieldsdef extract_arms_interventions_card(driver):    """Extract arms & interventions: experimental/control groups."""    fields = []        try:        # Look for Arms & Interventions section        arms_section = driver.find_elements(By.XPATH, "//h1[text()='Arms & Interventions']//following-sibling::div")                if arms_section:            arm_cards = arms_section[0].find_elements(By.CSS_SELECTOR, "div.border-\\[1px\\]")                        for idx, card in enumerate(arm_cards):                # Get arm type and description                try:                    arm_text = card.find_element(By.TAG_NAME, "p").text.strip()                    fields.append({                        'card': 'arms_interventions',                        'category': 'study_arm',                        'label': f'Study Arm {idx + 1}',                        'value': arm_text[:300] if len(arm_text) > 300 else arm_text,                        'selector': get_css_selector(card, driver),                        'element_type': 'div',                        'has_data': True                    })                except:                    continue    except Exception as e:        print(f"    Warning: Error extracting arms & interventions card: {e}")        return fieldsdef extract_study_measures_card(driver):    """Extract study measures: primary and secondary outcomes."""    fields = []        try:        # Primary Outcomes        primary_outcomes = driver.find_elements(By.XPATH, "//h3[contains(text(), 'PRIMARY OUTCOME')]//following-sibling::div//div[contains(@class, 'border')]")                for idx, outcome in enumerate(primary_outcomes[:3]):  # Limit to first 3            try:                measure = outcome.find_element(By.XPATH, ".//h1[text()='Outcome Measure']//following-sibling::p").text.strip()                timeframe = outcome.find_element(By.XPATH, ".//h1[text()='Time Frame']//following-sibling::p").text.strip()                                fields.append({                    'card': 'study_measures',                    'category': 'primary_outcome',                    'label': f'Primary Outcome {idx + 1}',                    'value': f"{measure[:200]} | Time Frame: {timeframe}",                    'selector': get_css_selector(outcome, driver),                    'element_type': 'div',                    'has_data': True                })            except:                continue                # Secondary Outcomes (sample first few)        secondary_outcomes = driver.find_elements(By.XPATH, "//h3[contains(text(), 'SECONDARY OUTCOME')]//following-sibling::div//div[contains(@class, 'border')]")                for idx, outcome in enumerate(secondary_outcomes[:3]):  # Limit to first 3            try:                measure = outcome.find_element(By.XPATH, ".//h1[text()='Outcome Measure']//following-sibling::p").text.strip()                timeframe = outcome.find_element(By.XPATH, ".//h1[text()='Time Frame']//following-sibling::p").text.strip()                                fields.append({                    'card': 'study_measures',                    'category': 'secondary_outcome',                    'label': f'Secondary Outcome {idx + 1}',                    'value': f"{measure[:200]} | Time Frame: {timeframe}",                    'selector': get_css_selector(outcome, driver),                    'element_type': 'div',                    'has_data': True                })            except:                continue                        # Add count of total outcomes        fields.append({            'card': 'study_measures',            'category': 'outcome_count',            'label': 'Total Secondary Outcomes',            'value': str(len(secondary_outcomes)),            'selector': 'N/A',            'element_type': 'computed',            'has_data': True        })            except Exception as e:        print(f"    Warning: Error extracting study measures card: {e}")        return fieldsdef extract_atoms_card(driver):    """Extract atoms card: related hypotheses/works count and samples."""    fields = []        try:        # Get atoms count        atoms_header = driver.find_element(By.XPATH, "//span[text()='Atoms']//following-sibling::span")        atoms_count = atoms_header.text.strip()                fields.append({            'card': 'atoms',            'category': 'atoms_count',            'label': 'Total Atoms',            'value': atoms_count,            'selector': get_css_selector(atoms_header, driver),            'element_type': 'span',            'has_data': True        })                # Get sample atom titles        atom_cards = driver.find_elements(By.XPATH, "//div[contains(@class, 'shadow-custom')]//a[contains(@href, '/hypothesis/')]")                for idx, atom in enumerate(atom_cards[:3]):  # Sample first 3            fields.append({                'card': 'atoms',                'category': 'related_hypothesis',                'label': f'Related Hypothesis {idx + 1}',                'value': atom.text.strip()[:200],                'selector': get_css_selector(atom, driver),                'element_type': 'a',                'has_data': True            })            except Exception as e:        print(f"    Warning: Error extracting atoms card: {e}")        return fieldsdef extract_contributing_orgs_card(driver):    """Extract contributing organizations card: sponsors and investigators."""    fields = []        try:        # Look for Contributing Organizations section        org_elements = driver.find_elements(By.XPATH, "//h1[text()='Contributing Organizations']//following-sibling::div//span")                for elem in org_elements:            text = elem.text.strip()            if text and ':' not in text and len(text) > 2:  # Filter out labels                # Determine role based on preceding text                try:                    parent_text = elem.find_element(By.XPATH, "./parent::div").text                    if 'Lead Sponsor' in parent_text:                        role = 'Lead Sponsor'                    elif 'Primary Investigator' in parent_text:                        role = 'Primary Investigator Organization'                    else:                        role = 'Contributing Organization'                                        fields.append({                        'card': 'contributing_organizations',                        'category': 'organization_role',                        'label': role,                        'value': text,                        'selector': get_css_selector(elem, driver),                        'element_type': 'span',                        'has_data': True                    })                except:                    continue    except Exception as e:        print(f"    Warning: Error extracting contributing organizations card: {e}")        return fieldsdef extract_clinical_trial_cards(driver, page_url):    """Extract all clinical trial detail page cards with labels."""    all_fields = []        print(f"    Extracting clinical trial cards...")        # Extract each card    print(f"      - Header card...")    all_fields.extend(extract_header_card_metrics(driver))        print(f"      - Abstract card...")    all_fields.extend(extract_abstract_card(driver))        print(f"      - Timeline card...")    all_fields.extend(extract_timeline_card(driver))        print(f"      - Metadata card...")    all_fields.extend(extract_metadata_card(driver))        print(f"      - Study Design card...")    all_fields.extend(extract_study_design_card(driver))        print(f"      - Participation Criteria card...")    all_fields.extend(extract_participation_criteria_card(driver))        print(f"      - Arms & Interventions card...")    all_fields.extend(extract_arms_interventions_card(driver))        print(f"      - Study Measures card...")    all_fields.extend(extract_study_measures_card(driver))        print(f"      - Atoms card...")    all_fields.extend(extract_atoms_card(driver))        print(f"      - Contributing Organizations card...")    all_fields.extend(extract_contributing_orgs_card(driver))        # Add page context    for field in all_fields:        field['page_url'] = page_url        field['page_title'] = driver.title        field['page_type'] = 'clinical_trial_detail'        return all_fields

In [ ]:
def extract_metadata_fields(driver):
    """Extract metadata fields like Conditions, Intervention, Study Type, Phase, etc."""
    fields = []
    
    try:
        # Wait for metadata section to load
        WebDriverWait(driver, ELEMENT_WAIT).until(
            EC.presence_of_element_located((By.ID, "metadata-content"))
        )
        
        # Find all metadata field elements
        metadata_elements = driver.find_elements(By.CSS_SELECTOR, "div#metadata-content h1")
        
        for element in metadata_elements:
            text = element.text.strip()
            if ':' in text:
                label, value = text.split(':', 1)
                label = label.strip()
                value = value.strip()
                
                # Try to find the span containing the value
                try:
                    value_span = element.find_element(By.TAG_NAME, "span")
                    value = value_span.text.strip()
                except:
                    pass
                
                fields.append({
                    'category': 'metadata',
                    'label': label,
                    'value': value,
                    'selector': get_css_selector(element, driver),
                    'element_type': 'h1',
                    'has_data': bool(value)
                })
    except TimeoutException:
        pass  # No metadata section on this page
    except Exception as e:
        print(f"    Warning: Could not extract metadata fields: {e}")
    
    return fields


def extract_button_metrics(driver):
    """Extract metrics from buttons like citation counts, etc."""
    fields = []
    
    try:
        # Find all buttons with aria-label (these often contain metrics)
        buttons = driver.find_elements(By.CSS_SELECTOR, "button[aria-label]")
        
        for button in buttons:
            aria_label = button.get_attribute('aria-label')
            text = button.text.strip()
            
            # Check if button contains numerical data
            if text and (text.isdigit() or re.search(r'\d+', text)):
                fields.append({
                    'category': 'button_metric',
                    'label': aria_label,
                    'value': text,
                    'selector': get_css_selector(button, driver),
                    'element_type': 'button',
                    'has_data': True
                })
    except Exception as e:
        print(f"    Warning: Could not extract button metrics: {e}")
    
    return fields


def extract_date_fields(driver):
    """Extract date fields from the page."""
    fields = []
    
    try:
        # Look for spans containing calendar icons and dates
        date_elements = driver.find_elements(By.XPATH, 
            "//span[contains(@class, 'flex') and .//svg and text()]")
        
        for element in date_elements:
            text = element.text.strip()
            # Check if it looks like a year or date
            if re.match(r'^\d{4}$', text) or re.match(r'\d{1,2}/\d{1,2}/\d{2,4}', text):
                fields.append({
                    'category': 'date',
                    'label': 'Date/Year',
                    'value': text,
                    'selector': get_css_selector(element, driver),
                    'element_type': 'span',
                    'has_data': True
                })
    except Exception as e:
        print(f"    Warning: Could not extract date fields: {e}")
    
    return fields


def extract_table_data(driver):
    """Extract data from tables if present."""
    fields = []
    
    try:
        tables = driver.find_elements(By.TAG_NAME, "table")
        
        for idx, table in enumerate(tables):
            # Get table headers
            headers = []
            try:
                header_cells = table.find_elements(By.TAG_NAME, "th")
                headers = [h.text.strip() for h in header_cells]
            except:
                pass
            
            # Get first data row as sample
            try:
                rows = table.find_elements(By.TAG_NAME, "tr")
                if len(rows) > 1:
                    cells = rows[1].find_elements(By.TAG_NAME, "td")
                    for cell_idx, cell in enumerate(cells):
                        label = headers[cell_idx] if cell_idx < len(headers) else f"Column {cell_idx + 1}"
                        value = cell.text.strip()
                        
                        fields.append({
                            'category': 'table',
                            'label': f"Table {idx + 1} - {label}",
                            'value': value[:100],
                            'selector': get_css_selector(cell, driver),
                            'element_type': 'td',
                            'has_data': bool(value)
                        })
            except:
                pass
    except Exception as e:
        print(f"    Warning: Could not extract table data: {e}")
    
    return fields


def extract_data_testid_fields(driver):
    """Extract all elements with data-testid attributes."""
    fields = []
    
    try:
        # Use BeautifulSoup for additional parsing
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Find all elements with data attributes
        data_elements = soup.find_all(attrs={'data-testid': True})
        for elem in data_elements:
            testid = elem.get('data-testid')
            text = elem.get_text(strip=True)
            if text:
                fields.append({
                    'category': 'data_testid',
                    'label': testid,
                    'value': text[:100],  # Limit to 100 chars
                    'selector': f'[data-testid="{testid}"]',
                    'element_type': elem.name,
                    'has_data': True
                })
    except Exception as e:
        print(f"    Warning: Could not extract data-testid fields: {e}")
    
    return fields


def extract_labeled_fields(driver):
    """Extract fields that have label-value patterns."""
    fields = []
    
    try:
        # Look for common label-value patterns in the DOM
        # Pattern 1: Elements with class containing 'label' followed by value
        label_elements = driver.find_elements(By.CSS_SELECTOR, "[class*='label'], [class*='Label']")
        
        for label_elem in label_elements:
            try:
                label_text = label_elem.text.strip()
                if label_text and len(label_text) < 100:
                    # Try to find adjacent value element
                    parent = label_elem.find_element(By.XPATH, "..")
                    parent_text = parent.text.strip()
                    
                    # Extract value (text after label)
                    if parent_text.startswith(label_text):
                        value = parent_text[len(label_text):].strip()
                        if value.startswith(':'):
                            value = value[1:].strip()
                        
                        if value:
                            fields.append({
                                'category': 'labeled_field',
                                'label': label_text,
                                'value': value[:100],
                                'selector': get_css_selector(label_elem, driver),
                                'element_type': label_elem.tag_name,
                                'has_data': True
                            })
            except:
                continue
        
        # Pattern 2: dt/dd pairs
        dt_elements = driver.find_elements(By.TAG_NAME, "dt")
        for dt in dt_elements:
            try:
                label = dt.text.strip()
                dd = dt.find_element(By.XPATH, "following-sibling::dd")
                value = dd.text.strip()
                
                if label and value:
                    fields.append({
                        'category': 'labeled_field',
                        'label': label,
                        'value': value[:100],
                        'selector': get_css_selector(dt, driver),
                        'element_type': 'dt',
                        'has_data': True
                    })
            except:
                continue
                
    except Exception as e:
        print(f"    Warning: Could not extract labeled fields: {e}")
    
    return fields


def extract_all_fields(driver, page_url, tab_name=None):
    """Extract all fields from the current page view.
    
    Args:
        driver: Selenium WebDriver instance
        page_url: URL of the current page
        tab_name: Optional name of the current tab (if on a tab)
    
    Returns:
        List of field dictionaries with metadata
    """
    all_fields = []
    
    # Extract different types of fields
    all_fields.extend(extract_metadata_fields(driver))
    all_fields.extend(extract_button_metrics(driver))
    all_fields.extend(extract_date_fields(driver))
    all_fields.extend(extract_table_data(driver))
    all_fields.extend(extract_data_testid_fields(driver))
    all_fields.extend(extract_labeled_fields(driver))
    
    # Add context to each field
    page_type = get_page_type(page_url)
    page_title = driver.title
    
    for field in all_fields:
        field['page_url'] = page_url
        field['page_title'] = page_title
        field['page_type'] = page_type
        field['tab_name'] = tab_name
    
    return all_fields


In [ ]:
def crawl_page_with_tabs(driver, url, depth=0):
    """Crawl a single page including all its tabs."""
    page_data = {
        'url': url,
        'page_type': get_page_type(url),
        'title': '',
        'depth': depth,
        'tabs': [],
        'fields': [],
        'links': set(),
        'crawl_timestamp': datetime.now().isoformat()
    }
    
    try:
        print(f"  {'  ' * depth}Crawling: {url}")
        driver.get(url)
        wait_for_page_load(driver)
        
        page_data['title'] = driver.title
        
        # Extract fields from main page

        # Check if this is a clinical trial detail page
        is_clinical_trial = 'clinical-trial' in url and 'ASC-CT-' in url

        if is_clinical_trial:
            # Use specialized clinical trial card extraction
            print(f"  {'  ' * depth}  - Detected clinical trial detail page, using card extraction")
            trial_fields = extract_clinical_trial_cards(driver, url)
            page_data['fields'].extend(trial_fields)
        else:
            # Extract fields from main page using generic extraction
            print(f"  {'  ' * depth}  - Extracting fields from main page (generic)")
            main_fields = extract_all_fields(driver, url)
            page_data['fields'].extend(main_fields)
        print(f"  {'  ' * depth}  - Extracting fields from main page")
        
        # Discover tabs (for detail pages)
        tabs = discover_tabs(driver)
        if tabs:
            print(f"  {'  ' * depth}  - Found {len(tabs)} tabs: {', '.join([t['name'] for t in tabs])}")
            
            for tab in tabs:
                try:
                    print(f"  {'  ' * depth}    → Clicking tab: {tab['name']}")
                    tab['element'].click()
                    time.sleep(TAB_SWITCH_WAIT)
                    
                    # Extract fields from this tab
                    tab_fields = extract_all_fields(driver, url, tab_name=tab['name'])
                    page_data['fields'].extend(tab_fields)
                    page_data['tabs'].append(tab['name'])
                    
                    print(f"  {'  ' * depth}      Extracted {len(tab_fields)} fields")
                except Exception as e:
                    print(f"  {'  ' * depth}      Warning: Could not process tab {tab['name']}: {e}")
                    continue
        
        # Discover links
        links = discover_links(driver)
        page_data['links'] = links
        print(f"  {'  ' * depth}  - Found {len(links)} links")
        
        print(f"  {'  ' * depth}✓ Total fields extracted: {len(page_data['fields'])}")
        
    except Exception as e:
        print(f"  {'  ' * depth}✗ Error crawling {url}: {e}")
    
    return page_data


def crawl_application(driver, search_queries, additional_urls=[], max_pages=MAX_PAGES_TO_CRAWL, max_depth=MAX_DEPTH):
    """Crawl the entire application starting from search queries and additional URLs."""
    
    visited = set()
    to_visit = deque()
    
    all_pages = []
    all_fields = []
    sitemap = defaultdict(list)  # parent_url -> [child_urls]
    
    page_count = 0
    
    print(f"\n{'='*60}")
    print(f"PHASE 1: SEARCH-BASED DISCOVERY")
    print(f"{'='*60}\n")
    
    # First, crawl all search queries
    for query in search_queries:
        if page_count >= max_pages:
            break
        
        print(f"\n[Search Query: '{query}']")
        
        try:
            # Crawl search results for this query
            search_pages, discovered_links = crawl_search_results(driver, query, depth=0)
            
            # Add search pages to results
            all_pages.extend(search_pages)
            for page in search_pages:
                all_fields.extend(page['fields'])
                page_count += 1
            
            # Add discovered links to queue for detailed crawling
            for link in discovered_links:
                if link not in visited:
                    to_visit.append((link, 1))  # depth 1 since they're from search
                    sitemap[search_pages[0]['url']].append(link)
            
            print(f"  Added {len(discovered_links)} detail pages to queue")
            
        except Exception as e:
            print(f"  ✗ Error processing search query '{query}': {e}")
            continue
    
    # Add any additional URLs to the queue
    for url in additional_urls:
        if url not in visited:
            to_visit.append((url, 0))
    
    print(f"\n{'='*60}")
    print(f"PHASE 2: DETAIL PAGE CRAWLING")
    print(f"{'='*60}\n")
    print(f"Queue: {len(to_visit)} pages to crawl\n")
    
    # Now crawl discovered detail pages
    while to_visit and page_count < max_pages:
        url, depth = to_visit.popleft()
        
        # Skip if already visited or too deep
        if url in visited or depth > max_depth:
            continue
        
        visited.add(url)
        page_count += 1
        
        print(f"\n[{page_count}/{max_pages}] Depth {depth}")
        
        # Crawl the page
        page_data = crawl_page_with_tabs(driver, url, depth)
        all_pages.append(page_data)
        all_fields.extend(page_data['fields'])
        
        # Add discovered links to queue (but limit based on page type)
        page_type = page_data['page_type']
        
        # For detail pages, follow related links (but limit them)
        if 'detail' in page_type:
            detail_links = [link for link in page_data['links'] 
                          if any(pattern in link for pattern in ['ASC-CT-', 'ASC-WK-', 'ASC-PT-', 'ASC-HY-'])]
            links_to_follow = detail_links[:3]  # Max 3 related pages per detail page
        else:
            links_to_follow = []
        
        for link in links_to_follow:
            if link not in visited:
                to_visit.append((link, depth + 1))
                sitemap[url].append(link)
        
        # Show progress
        print(f"  Queue size: {len(to_visit)} | Visited: {len(visited)}")
    
    print(f"\n{'='*60}")
    print(f"CRAWL COMPLETE!")
    print(f"  Pages crawled: {page_count}")
    print(f"  Total fields extracted: {len(all_fields)}")
    print(f"{'='*60}")
    
    return {
        'pages': all_pages,
        'fields': all_fields,
        'sitemap': dict(sitemap),
        'visited_urls': list(visited)
    }

## Page Crawling Functions

In [ ]:
# Setup and login
print("Setting up crawler...\n")
driver = setup_driver(headless=False)

# Login
login_success = login_to_application(driver, LOGIN_URL, SUPABASE_EMAIL, SUPABASE_PASSWORD)

if not login_success:
    print("Login failed. Please check credentials.")
    driver.quit()
else:
    print("\nStarting application crawl...\n")
    print(f"{'='*60}")
    print(f"Configuration:")
    print(f"  Max pages: {MAX_PAGES_TO_CRAWL}")
    print(f"  Max depth: {MAX_DEPTH}")
    print(f"  Results per search tab: {MAX_RESULTS_PER_SEARCH_TAB}")
    print(f"  Search queries: {len(SEARCH_QUERIES)}")
    for query in SEARCH_QUERIES:
        print(f"    - '{query}'")
    if ADDITIONAL_URLS:
        print(f"  Additional URLs: {len(ADDITIONAL_URLS)}")
        for url in ADDITIONAL_URLS:
            print(f"    - {url}")
    print(f"{'='*60}\n")
    
    # Crawl
    crawl_results = crawl_application(driver, SEARCH_QUERIES, ADDITIONAL_URLS)
    
    # Cleanup
    driver.quit()

## Run the Crawler

In [ ]:
# Note: Use Cell 14 above to run the crawler.
# This cell was a duplicate with incorrect variable references and has been cleared.

## Generate Reports

In [ ]:
# Create DataFrames
df_fields = pd.DataFrame(crawl_results['fields'])
df_pages = pd.DataFrame([{
    'url': p['url'],
    'page_type': p['page_type'],
    'title': p['title'],
    'depth': p['depth'],
    'num_tabs': len(p['tabs']),
    'tabs': ', '.join(p['tabs']),
    'num_fields': len(p['fields']),
    'num_links': len(p['links']),
} for p in crawl_results['pages']])

print("\n" + "="*60)
print("CRAWL SUMMARY")
print("="*60)
print(f"\nTotal Pages: {len(df_pages)}")
print(f"Total Fields: {len(df_fields)}")
print(f"\nPages by Type:")
print(df_pages['page_type'].value_counts())
print(f"\nFields by Category:")
print(df_fields['category'].value_counts())
print(f"\nTop 10 Pages by Field Count:")
print(df_pages.nlargest(10, 'num_fields')[['url', 'page_type', 'num_fields', 'num_tabs']])

## Application Structure Map

In [ ]:
# Generate application structure
print("\n" + "="*60)
print("APPLICATION STRUCTURE MAP")
print("="*60)

for page in crawl_results['pages'][:20]:  # Show first 20
    indent = "  " * page['depth']
    print(f"\n{indent}📄 {page['url'].replace(BASE_URL, '')}")
    print(f"{indent}   Type: {page['page_type']} | Fields: {len(page['fields'])} | Links: {len(page['links'])}")
    if page['tabs']:
        print(f"{indent}   Tabs: {', '.join(page['tabs'])}")

## Field Coverage Analysis

In [ ]:
# Analyze field coverage
print("\n" + "="*60)
print("FIELD COVERAGE BY PAGE TYPE")
print("="*60)

coverage = df_fields.groupby(['page_type', 'category']).agg({
    'label': 'count',
    'has_data': 'sum'
}).rename(columns={'label': 'total_fields', 'has_data': 'fields_with_data'})

print(coverage)

# Show unique field labels by page type
print("\n" + "="*60)
print("UNIQUE FIELD LABELS BY PAGE TYPE")
print("="*60)

for page_type in df_fields['page_type'].unique():
    print(f"\n{page_type.upper()}:")
    type_fields = df_fields[df_fields['page_type'] == page_type]
    unique_labels = type_fields['label'].unique()[:15]
    for label in unique_labels:
        print(f"  - {label}")
    if len(type_fields['label'].unique()) > 15:
        print(f"  ... and {len(type_fields['label'].unique()) - 15} more")

## Tab Analysis

In [ ]:
# Analyze tabs
print("\n" + "="*60)
print("TAB ANALYSIS")
print("="*60)

tab_fields = df_fields[df_fields['tab_name'].notna()]
if len(tab_fields) > 0:
    print(f"\nTotal fields from tabs: {len(tab_fields)}")
    print(f"\nFields by Tab:")
    print(tab_fields.groupby('tab_name')['label'].count().sort_values(ascending=False))
    
    print(f"\nPages with Tabs:")
    pages_with_tabs = df_pages[df_pages['num_tabs'] > 0]
    print(pages_with_tabs[['url', 'page_type', 'tabs', 'num_tabs']].to_string())
else:
    print("\nNo tabs were detected on any pages.")

## Export Results

In [ ]:
# Export all data
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Field mapping CSV
field_file = f'field_mapping_full_{timestamp}.csv'
df_fields.to_csv(field_file, index=False)
print(f"\n✓ Field mapping exported to: {field_file}")

# 2. Page structure CSV
page_file = f'page_structure_{timestamp}.csv'
df_pages.to_csv(page_file, index=False)
print(f"✓ Page structure exported to: {page_file}")

# 3. Complete crawl data JSON
json_file = f'crawl_results_{timestamp}.json'
with open(json_file, 'w') as f:
    json.dump({
        'pages': [{
            **p,
            'links': list(p['links'])  # Convert sets to lists for JSON
        } for p in crawl_results['pages']],
        'sitemap': crawl_results['sitemap'],
        'visited_urls': crawl_results['visited_urls'],
        'summary': {
            'total_pages': len(crawl_results['pages']),
            'total_fields': len(crawl_results['fields']),
            'crawl_timestamp': timestamp
        }
    }, f, indent=2)
print(f"✓ Complete crawl data exported to: {json_file}")

# 4. Sitemap visualization
sitemap_file = f'sitemap_{timestamp}.txt'
with open(sitemap_file, 'w') as f:
    f.write("APPLICATION SITEMAP\n")
    f.write("="*80 + "\n\n")
    
    for page in crawl_results['pages']:
        indent = "  " * page['depth']
        f.write(f"{indent}{page['url']}\n")
        f.write(f"{indent}  Type: {page['page_type']}\n")
        f.write(f"{indent}  Fields: {len(page['fields'])}\n")
        if page['tabs']:
            f.write(f"{indent}  Tabs: {', '.join(page['tabs'])}\n")
        f.write("\n")

print(f"✓ Sitemap exported to: {sitemap_file}")

print(f"\n{'='*60}")
print("All exports complete!")
print(f"{'='*60}")